# Bhojpuri BPE Tokenizer Training

## What is a Tokenizer?

A **tokenizer** is a tool that splits text into smaller pieces called **tokens**. 

### Example:
```
Sentence: "यह एक परीक्षण वाक्य है।"
                      ↓
Tokens:   ["य", "ह", "एक", "परीक्षण", "वाक्य", "है", "।"]
```

## BPE (Byte-Pair Encoding) Tokenizer

BPE learns **which character sequences appear frequently together** and creates tokens from them.

### Training Process (What happens):

1. **Read all training text** from files (bhoj.txt from train/ and val/ folders)
2. **Find common byte patterns** - Which character pairs appear most often?
3. **Merge common pairs into tokens** - Turn "य" + "ह" into "यह" token
4. **Repeat** until reaching target vocab size (16,000 tokens for Bhojpuri)
5. **Save the learned vocab** as `bhoj_tokenizer.json`

### At inference (using the tokenizer):
```
Input:  "यह एक परीक्षण"
        ↓
Output: [<token_id_1>, <token_id_2>, <token_id_3>, ...]
        ↓ (model processes these token IDs)
```

## This Notebook's Steps:

1. **Setup**: Define data paths and config
2. **Utilities**: Vocabulary size calculation (dynamic heuristic)
3. **Training**: Run BPE on train+val corpus
4. **Evaluation**: Test on held-out test set
5. **Regression**: Verify combining marks (matra/virama) survive correctly


## ⚠️ Important: Trained from Scratch (No Pretrained Tokenizers)

**This notebook trains a BPE tokenizer from scratch on the project corpus.**

- Uses the standalone `tokenizers` library (NOT `transformers`)
- Starts with a blank `models.BPE()` with no pretrained vocabulary
- `initial_alphabet=ByteLevel.alphabet()` only seeds the 256 raw byte symbols
- **No `.from_pretrained()` call anywhere** — all merges/vocab learned purely from your Bhojpuri corpus
- Satisfies the project constraint: *No pretrained models, no pretrained tokenizers*


In [1]:
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Optional

from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders, processors, trainers

# ============================================================================
# ⚙️ CONFIGURATION: DATA ROOT PATH - MODIFY THIS IF DATA IS IN A DIFFERENT LOCATION
# ============================================================================
# Default: assumes notebook is in bhojpuri/tokenizer/ and data is in bhojpuri/data/
notebook_dir = Path("/kaggle/working/")
DATA_ROOT = Path("/kaggle/input/datasets/kspsvlnsiddardha/lma-slm/bhojpuri/data")  # -> .../bhojpuri/data

# If data is elsewhere, set it explicitly:
# DATA_ROOT = Path("/kaggle/input/bhojpuri-data")  # Example for Kaggle

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"
TOKENIZER_DIR = notebook_dir  # -> .../bhojpuri/tokenizer

print(f"✓ Data root: {DATA_ROOT}")
print(f"✓ Tokenizer dir: {TOKENIZER_DIR}")

# ============================================================================
# Language & tokenizer config
# ============================================================================
LANG = "Bhojpuri"
LANG_SHORT = "bhoj"
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

print(f"\n🚀 Training {LANG} BPE Tokenizer")


✓ Data root: /kaggle/input/datasets/kspsvlnsiddardha/lma-slm/bhojpuri/data
✓ Tokenizer dir: /kaggle/working

🚀 Training Bhojpuri BPE Tokenizer


In [2]:
# ============================================================================
# ALL FUNCTION DEFINITIONS (Define everything FIRST before using)
# ============================================================================

# ---- Utilities ----
def estimate_corpus_tokens(total_bytes: int, bytes_per_token: float = 4.0) -> int:
    """Rough token estimate from corpus size (bytes/4 heuristic)."""
    return int(total_bytes / bytes_per_token)


def compute_vocab_size(total_tokens_estimate: int) -> int:
    """Tiered vocab size heuristic: <50M->8K, 50M-200M->16K, 200M-1B->32K, >=1B->50K"""
    if total_tokens_estimate < 50_000_000:
        return 8_000
    elif total_tokens_estimate < 200_000_000:
        return 12_000
    elif total_tokens_estimate < 1_000_000_000:
        return 32_000
    else:
        return 50_000

# ---- Corpus Discovery ----
def gather_training_files(split_dirs: list[Path]) -> list[Path]:
    """Find all *.txt files in given split directories, sorted."""
    files = []
    for split_dir in split_dirs:
        if split_dir.exists():
            files.extend(sorted(split_dir.glob("*.txt")))
    return files


def total_bytes(files: list[Path]) -> int:
    """Compute total size of files in bytes."""
    return sum(f.stat().st_size for f in files if f.exists())

# ---- Tokenizer Construction ----
def create_bpe_tokenizer() -> Tokenizer:
    """Create a ByteLevel BPE tokenizer with correct normalization."""
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
    return tokenizer


def build_trainer(vocab_size: int) -> trainers.BpeTrainer:
    """Build a BPE trainer with specified vocab size and special tokens."""
    return trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )

# ---- Evaluation ----
def evaluate_tokenizer(tokenizer: Tokenizer, test_files: list[Path], sample_lines: int = 500) -> dict:
    """Evaluate tokenizer on held-out test set."""
    import random
    logger.info("Evaluating tokenizer on held-out test set...")

    sampled_lines = []
    rng = random.Random(42)
    total_read = 0

    for test_file in test_files:
        if not test_file.exists():
            continue
        with open(test_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    continue
                total_read += 1
                if len(sampled_lines) < sample_lines:
                    sampled_lines.append(line)
                else:
                    j = rng.randint(0, total_read - 1)
                    if j < sample_lines:
                        sampled_lines[j] = line

    logger.info(f"Sampled {len(sampled_lines)} lines from {total_read} read")

    token_lengths = []
    char_counts = []
    token_counts = []
    unk_count = 0
    total_tokens = 0
    roundtrip_pass = 0
    example_triples = []

    for line in sampled_lines[:100]:
        encoded = tokenizer.encode(line)
        decoded = tokenizer.decode(encoded.ids)

        token_lengths.append(len(encoded.ids))
        char_counts.append(len(line))
        token_counts.append(len(encoded.ids))

        for token_id in encoded.ids:
            total_tokens += 1
            if token_id == UNK_ID:
                unk_count += 1

        if decoded == line:
            roundtrip_pass += 1

        if len(example_triples) < 3:
            example_triples.append({
                "original": line[:80],
                "num_tokens": len(encoded.ids),
                "roundtrip_ok": decoded == line,
            })

    avg_tokens_per_line = sum(token_lengths) / len(token_lengths) if token_lengths else 0
    avg_chars_per_token = sum(char_counts) / sum(token_counts) if token_counts else 0
    unk_rate = 100.0 * unk_count / total_tokens if total_tokens > 0 else 0
    roundtrip_rate = 100.0 * roundtrip_pass / len(sampled_lines) if sampled_lines else 0

    return {
        "samples_evaluated": len(sampled_lines),
        "avg_tokens_per_line": round(avg_tokens_per_line, 2),
        "avg_chars_per_token": round(avg_chars_per_token, 2),
        "unk_rate_percent": round(unk_rate, 4),
        "roundtrip_match_percent": round(roundtrip_rate, 1),
        "example_triples": example_triples,
    }

print("✅ ALL FUNCTIONS DEFINED - Ready to use!")

# ---- Full Tokenizer Report (entire test set) ----
def generate_tokenizer_report(tokenizer: Tokenizer, test_files: list[Path],
                               vocab_size_requested: int, top_n: int = 20) -> dict:
    """Full test-set pass: vocab size, token-frequency stats, avg chars/token,
    tokenization examples, UNK statistic — for the project report."""
    from collections import Counter

    token_freq = Counter()
    total_tokens = 0
    total_chars = 0
    total_lines = 0
    unk_count = 0

    for test_file in test_files:
        if not test_file.exists():
            continue
        with open(test_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    continue
                total_lines += 1
                total_chars += len(line)
                encoded = tokenizer.encode(line)
                total_tokens += len(encoded.ids)
                token_freq.update(encoded.ids)
                unk_count += sum(1 for tid in encoded.ids if tid == UNK_ID)

    vocab_actual = tokenizer.get_vocab_size()
    most_common = [
        {"token": tokenizer.decode([tid]), "id": tid, "count": count,
         "percent_of_tokens": round(100.0 * count / total_tokens, 4)}
        for tid, count in token_freq.most_common(top_n)
    ]
    freq_values = list(token_freq.values())
    unique_tokens_used = len(token_freq)

    return {
        "vocab_size_requested": vocab_size_requested,
        "vocab_size_actual": vocab_actual,
        "unique_tokens_used_in_test": unique_tokens_used,
        "vocab_coverage_percent": round(100.0 * unique_tokens_used / vocab_actual, 2),
        "total_test_lines": total_lines,
        "total_test_tokens": total_tokens,
        "avg_chars_per_token": round(total_chars / total_tokens, 4) if total_tokens else 0,
        "unk_count": unk_count,
        "unk_rate_percent": round(100.0 * unk_count / total_tokens, 4) if total_tokens else 0,
        "token_frequency_top_n": most_common,
        "token_frequency_stats": {
            "min": min(freq_values) if freq_values else 0,
            "max": max(freq_values) if freq_values else 0,
            "mean": round(sum(freq_values) / len(freq_values), 2) if freq_values else 0,
        },
    }


✅ ALL FUNCTIONS DEFINED - Ready to use!


In [3]:
# ============================================================================
# Discover corpus files
# ============================================================================

logger.info("Discovering corpus files...")
train_files = gather_training_files([TRAIN_DIR])
val_files = gather_training_files([VAL_DIR])
test_files = gather_training_files([TEST_DIR])

logger.info(f"Train files: {[f.name for f in train_files]}")
logger.info(f"Val files: {[f.name for f in val_files]}")
logger.info(f"Test files: {[f.name for f in test_files]}")

# Compute vocab size from train+val
train_val_files = train_files + val_files
train_val_bytes = total_bytes(train_val_files)
train_val_tokens = estimate_corpus_tokens(train_val_bytes)
vocab_size = compute_vocab_size(train_val_tokens)

print(f"\n📊 Corpus stats (train+val):")
print(f"  Total bytes: {train_val_bytes / (1024**2):.1f} MB")
print(f"  Estimated tokens: {train_val_tokens:,}")
print(f"  Vocab size (heuristic): {vocab_size:,}")


[INFO] Discovering corpus files...
[INFO] Train files: ['bhoj.txt']
[INFO] Val files: ['bhoj.txt']
[INFO] Test files: ['bhoj.txt']



📊 Corpus stats (train+val):
  Total bytes: 1001.4 MB
  Estimated tokens: 262,506,697
  Vocab size (heuristic): 32,000


In [4]:
# ============================================================================
# Train tokenizer
# ============================================================================

logger.info("Creating tokenizer...")
tokenizer = create_bpe_tokenizer()

logger.info("Building trainer...")
trainer = build_trainer(vocab_size)

logger.info("Training BPE (this may take a few minutes)...")
training_files = train_val_files
tokenizer.train(
    files=[str(f) for f in training_files],
    trainer=trainer,
)

print("✓ Training complete")


[INFO] Creating tokenizer...
[INFO] Building trainer...
[INFO] Training BPE (this may take a few minutes)...





✓ Training complete


In [5]:
# ============================================================================
# Save tokenizer and config
# ============================================================================

tokenizer_path = TOKENIZER_DIR / f"{LANG_SHORT}_tokenizer.json"
logger.info(f"Saving tokenizer to {tokenizer_path.name}...")
tokenizer.save(str(tokenizer_path))

vocab_actual = tokenizer.get_vocab_size()
logger.info(f"Vocab size actual: {vocab_actual:,}")
if vocab_actual != vocab_size:
    logger.warning(f"  Note: actual ({vocab_actual}) differs from requested ({vocab_size})")

# Save config
config = {
    "language": LANG,
    "tokenizer_type": "BPE",
    "model": "ByteLevel BPE",
    "normalizer": "NFC",
    "vocab_size_requested": vocab_size,
    "vocab_size_actual": vocab_actual,
    "special_tokens": SPECIAL_TOKENS,
    "created_at": datetime.now().isoformat(),
}

config_path = TOKENIZER_DIR / "tokenizer_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
logger.info(f"Saved config to {config_path.name}")

print(f"✓ Tokenizer and config saved")


[INFO] Saving tokenizer to bhoj_tokenizer.json...
[INFO] Vocab size actual: 32,000
[INFO] Saved config to tokenizer_config.json


✓ Tokenizer and config saved


In [6]:
# ============================================================================
# Generate and save comprehensive tokenizer report
# ============================================================================

logger.info("Generating comprehensive tokenizer report...")
report = generate_tokenizer_report(tokenizer, test_files, vocab_size)

print(f"\n📊 TOKENIZER REPORT:")
print(f"  Vocab size (requested): {report['vocab_size_requested']:,}")
print(f"  Vocab size (actual): {report['vocab_size_actual']:,}")
print(f"  Unique tokens used in test: {report['unique_tokens_used_in_test']:,}")
print(f"  Vocab coverage: {report['vocab_coverage_percent']:.2f}%")
print(f"  Test set: {report['total_test_lines']:,} lines, {report['total_test_tokens']:,} tokens")
print(f"  Avg chars/token: {report['avg_chars_per_token']:.4f}")
print(f"  UNK count: {report['unk_count']:,}")
print(f"  UNK rate: {report['unk_rate_percent']:.4f}%")

print(f"\n📈 Top-20 Most Frequent Tokens:")
for i, item in enumerate(report['token_frequency_top_n'][:20], 1):
    token_repr = repr(item['token']) if len(item['token']) <= 20 else repr(item['token'][:20] + '...')
    print(f"  {i:2d}. {token_repr:25s} id={item['id']:5d} count={item['count']:8d} ({item['percent_of_tokens']:.2f}%)")

report_path = TOKENIZER_DIR / f"bhoj_tokenizer_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
logger.info(f"Saved report to {report_path.name}")


[INFO] Generating comprehensive tokenizer report...
[INFO] Saved report to bhoj_tokenizer_report.json



📊 TOKENIZER REPORT:
  Vocab size (requested): 32,000
  Vocab size (actual): 32,000
  Unique tokens used in test: 29,261
  Vocab coverage: 91.44%
  Test set: 182,673 lines, 29,039,468 tokens
  Avg chars/token: 1.5602
  UNK count: 0
  UNK rate: 0.0000%

📈 Top-20 Most Frequent Tokens:
   1. 'ा'                       id=  263 count= 2824980 (9.73%)
   2. 'े'                       id=  264 count= 2077576 (7.15%)
   3. '्'                       id=  266 count= 1822975 (6.28%)
   4. 'ि'                       id=  268 count= 1339554 (4.61%)
   5. ' क'                      id=  267 count= 1234270 (4.25%)
   6. 'र'                       id=  265 count=  820657 (2.83%)
   7. 'ी'                       id=  272 count=  800498 (2.76%)
   8. 'ो'                       id=  276 count=  679734 (2.34%)
   9. 'त'                       id=  271 count=  574703 (1.98%)
  10. ' स'                      id=  275 count=  541131 (1.86%)
  11. 'ु'                       id=  285 count=  483948 (1.67%)
  12. 'ल'   

## Evaluation on Test Set


In [7]:
# ============================================================================
# Evaluate tokenizer
# ============================================================================

import random

def evaluate_tokenizer(tokenizer: Tokenizer, test_files: list[Path], sample_lines: int = 500) -> dict:
    """Evaluate tokenizer on held-out test set."""
    logger.info("Evaluating tokenizer on held-out test set...")

    sampled_lines = []
    rng = random.Random(42)
    total_read = 0

    for test_file in test_files:
        if not test_file.exists():
            continue
        with open(test_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    continue
                total_read += 1
                if len(sampled_lines) < sample_lines:
                    sampled_lines.append(line)
                else:
                    j = rng.randint(0, total_read - 1)
                    if j < sample_lines:
                        sampled_lines[j] = line

    logger.info(f"Sampled {len(sampled_lines)} lines from {total_read} read")

    token_lengths = []
    char_counts = []
    token_counts = []
    unk_count = 0
    total_tokens = 0
    roundtrip_pass = 0
    example_triples = []

    for line in sampled_lines[:100]:
        encoded = tokenizer.encode(line)
        decoded = tokenizer.decode(encoded.ids)

        token_lengths.append(len(encoded.ids))
        char_counts.append(len(line))
        token_counts.append(len(encoded.ids))

        for token_id in encoded.ids:
            total_tokens += 1
            if token_id == UNK_ID:
                unk_count += 1

        if decoded == line:
            roundtrip_pass += 1

        if len(example_triples) < 3:
            example_triples.append({
                "original": line[:80],
                "num_tokens": len(encoded.ids),
                "roundtrip_ok": decoded == line,
            })

    avg_tokens_per_line = sum(token_lengths) / len(token_lengths) if token_lengths else 0
    avg_chars_per_token = sum(char_counts) / sum(token_counts) if token_counts else 0
    unk_rate = 100.0 * unk_count / total_tokens if total_tokens > 0 else 0
    roundtrip_rate = 100.0 * roundtrip_pass / len(sampled_lines) if sampled_lines else 0

    return {
        "samples_evaluated": len(sampled_lines),
        "avg_tokens_per_line": round(avg_tokens_per_line, 2),
        "avg_chars_per_token": round(avg_chars_per_token, 2),
        "unk_rate_percent": round(unk_rate, 4),
        "roundtrip_match_percent": round(roundtrip_rate, 1),
        "example_triples": example_triples,
    }

eval_results = evaluate_tokenizer(tokenizer, test_files)

print(f"\n📈 Evaluation Results:")
print(f"  Samples evaluated: {eval_results['samples_evaluated']}")
print(f"  Avg tokens/line: {eval_results['avg_tokens_per_line']}")
print(f"  Avg chars/token: {eval_results['avg_chars_per_token']:.2f}")
print(f"  UNK rate: {eval_results['unk_rate_percent']:.4f}%")
print(f"  Roundtrip match: {eval_results['roundtrip_match_percent']:.1f}%")

print(f"\n📝 Example Encode/Decode:")
for i, triple in enumerate(eval_results['example_triples'], 1):
    print(f"  {i}. {triple['original'][:60]}...")
    print(f"     Tokens: {triple['num_tokens']}, Roundtrip OK: {triple['roundtrip_ok']}")


[INFO] Evaluating tokenizer on held-out test set...
[INFO] Sampled 500 lines from 182673 read



📈 Evaluation Results:
  Samples evaluated: 500
  Avg tokens/line: 159.48
  Avg chars/token: 1.55
  UNK rate: 0.0000%
  Roundtrip match: 20.0%

📝 Example Encode/Decode:
  1. काला ईंट पेंटाइल ईंट हवे जेवन लिम्बू पानी आ रंगीन रंग के मिश...
     Tokens: 51, Roundtrip OK: True
  2. कारिका का आशय पढ़े है अविवद्धितवाज्य नामक ध्यान 74 और वाक्य ...
     Tokens: 901, Roundtrip OK: True
  3. आनंद जिला भारत के गुजरात राज्य में एगो जिला बाटे।...
     Tokens: 34, Roundtrip OK: True


In [8]:
# ============================================================================
# Regression tests (combining marks: vowel signs, virama)
# ============================================================================

test_cases = [
    ("क्ष", "Devanagari conjunct (virama)"),
    ("कि", "Devanagari vowel sign ि (U+093F)"),
    ("की", "Devanagari vowel sign ी (U+0940)"),
    ("म्य", "Devanagari conjunct (m + virama + y)"),
]

print("\n🔍 Regression Tests (combining marks):")
all_pass = True
for text, description in test_cases:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)
    passed = (decoded == text)
    status = "✓" if passed else "✗"
    print(f"  {status} {description}")
    print(f"     Input: {text}, Decoded: {decoded}")
    if not passed:
        all_pass = False

if all_pass:
    print("\n✓ All regression tests PASSED!")
else:
    print("\n✗ Some regression tests FAILED (check normalizer)")



🔍 Regression Tests (combining marks):
  ✓ Devanagari conjunct (virama)
     Input: क्ष, Decoded: क्ष
  ✓ Devanagari vowel sign ि (U+093F)
     Input: कि, Decoded: कि
  ✓ Devanagari vowel sign ी (U+0940)
     Input: की, Decoded: की
  ✓ Devanagari conjunct (m + virama + y)
     Input: म्य, Decoded: म्य

✓ All regression tests PASSED!


In [9]:
# ============================================================================
# Test cases: combining marks + FULL SENTENCES (sentence-level tokenization)
# ============================================================================

print("\n" + "="*70)
print("TEST CASES: SENTENCE-LEVEL TOKENIZATION")
print("="*70)

# Test 1: Combining marks regression (matra/virama)
print("\n🔍 Regression Tests (Combining Marks - Matra/Virama):") 
combining_test_cases = [
    ("क्ष", "Devanagari conjunct (virama)"),
    ("कि", "Devanagari vowel sign ि (U+093F)"),
    ("की", "Devanagari vowel sign ी (U+0940)"),
    ("म्य", "Devanagari conjunct (m + virama + y)"),
]

all_pass = True
for text, description in combining_test_cases:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)
    passed = (decoded == text)
    status = "✓" if passed else "✗"
    print(f"  {status} {description}")
    print(f"     Input: {text}, Decoded: {decoded}")
    if not passed:
        all_pass = False

if all_pass:
    print("\n  ✓ All combining mark tests PASSED!")
else:
    print("\n  ✗ Some tests FAILED")

# Test 2: Full sentence tokenization
print("\n📝 Sentence-Level Tokenization Examples:")
sentence_test_cases = [
    "यह एक परीक्षण वाक्य है।",
    "भारत एक महान देश है।",
    "हिन्दी भाषा बहुत सुंदर है।",
    "मैं एक विद्यार्थी हूँ।",
]

for sentence in sentence_test_cases:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded.ids)
    # Decode each token individually to show proper characters
    token_chars = [tokenizer.decode([tid]) for tid in encoded.ids]
    match = "✓" if decoded == sentence else "✗"
    print(f"\n  {match} Sentence: {sentence[:50]}...")
    print(f"     Tokens (decoded): {' | '.join(token_chars[:20])}{'...' if len(token_chars) > 20 else ''}")
    print(f"     Token count: {len(encoded.ids)}")
    print(f"     Token IDs: {encoded.ids[:15]}{'...' if len(encoded.ids) > 15 else ''}")
    print(f"     Roundtrip OK: {decoded == sentence}")



TEST CASES: SENTENCE-LEVEL TOKENIZATION

🔍 Regression Tests (Combining Marks - Matra/Virama):
  ✓ Devanagari conjunct (virama)
     Input: क्ष, Decoded: क्ष
  ✓ Devanagari vowel sign ि (U+093F)
     Input: कि, Decoded: कि
  ✓ Devanagari vowel sign ी (U+0940)
     Input: की, Decoded: की
  ✓ Devanagari conjunct (m + virama + y)
     Input: म्य, Decoded: म्य

  ✓ All combining mark tests PASSED!

📝 Sentence-Level Tokenization Examples:

  ✓ Sentence: यह एक परीक्षण वाक्य है।...
     Tokens (decoded): यह |  एक |  पर | ी | क | ् | षण |  व | ा | क | ् | य |  ह | ै।
     Token count: 14
     Token IDs: [1061, 373, 324, 272, 273, 266, 447, 302, 263, 273, 266, 279, 286, 441]
     Roundtrip OK: True

  ✓ Sentence: भारत एक महान देश है।...
     Tokens (decoded): भ | ा | रत |  एक |  मह | ा | न |  द | े | श |  ह | ै।
     Token count: 12
     Token IDs: [322, 263, 353, 373, 411, 263, 269, 299, 264, 305, 286, 441]
     Roundtrip OK: True

  ✓ Sentence: हिन्दी भाषा बहुत सुंदर है।...
     Tokens (decod